## Overview

This notebook loads and visualizes the results produced by `train_and_FIM_biased.py` and `train_and_FIM_UNbiased.py` for the 2D, partially-tensorized Fourier regression models (V represented as an MPS, U dense, local parameter basis dimension Dloc=3), comparing "full" model (all singular values ~equal, high effective dimension) against "cutoff" model (correlation spectrum truncated/decayed after index `cutoff`, low effective dimension) pairs. It reproduces the paper's central comparison (cf. Fig. 4): MSE training curves for the full vs. cutoff model, both for the biased data-generating scenario (the data-generating function lies exactly in the cutoff model's expressible function space) and the unbiased scenario (the data is generated independently of the trained models). It then plots Delta_{f-c}MSE_min = MSE_full_min - MSE_cut_min against the effective-dimension gap between the full and cutoff models, for the biased and unbiased cases overlaid, illustrating that low effective dimension helps training for biased models (Delta > 0) while high effective dimension helps unbiased models (Delta < 0).

Imports the packages and custom modules (FIM and training helper functions) needed to load and plot the saved results.

In [ ]:
# Importing necessary packages
import sys
import os
import copy
import importlib
import re
import pickle

import pennylane.numpy as np



import matplotlib
import matplotlib.pyplot as plt



# Current path for importing custom functions
path_base = '/home/b/b309245/cloud_cover_qml_parametrization/DYAMOND_cellbased_implementations/'
sys.path.insert(0, path_base + 'qml_useful_functions')

import qnn_layouts_pennylane
importlib.reload(qnn_layouts_pennylane)

import FIM_functions_jax
importlib.reload(FIM_functions_jax)

import training_functions_jax
importlib.reload(training_functions_jax)

Sets the results folder and the model/training hyperparameters (no_of_features, max_frequency, no_params, bond_dim, cutoff, dim_basis_single_param, learning_rate, batch_size) and the decay_exp_vec grid used when generating the biased and unbiased results, so the result filenames can be reconstructed below.

In [ ]:
### ---------------------------------------------------------------------------------------- ###
## ------------------------------------ Basic model specs ----------------------------------- ##
### ---------------------------------------------------------------------------------------- ###

# Folder in which to load data from
results_folder = '/work/bd1179/b309245/fourier_models_train_and_FIM/train_and_FIM_2D_TN_Dloc3/'

### 'biased_data_gen': the data generating funct. is fully encompassed by both full and cutoff models
### 'UNbiased_data_gen': the data generating funct. is random, not encompassed by full and cutoff models

name_data_gen_fullbiased = 'biased_data_gen'
name_data_gen_unbiased = 'UNbiased_data_gen'

# Learning rate
learning_rate = 0.02
learning_rate_name = '0p02'

# Batch size for training
batch_size = 5
batch_size_name = str(batch_size)

### Decay exponents for cutoff model
decay_exp_vec = [0.05, 0.07, 0.1, 0.2, 0.333, 0.5, 0.7, 1.0, 2.0, 3.33]
decay_exp_name_vec = ['0p05', '0p07', '0p1', '0p2', '0p333', '0p5', '0p7', '1p0', '2p0', '3p33']

### No. of features (dimension of input vectors)
no_of_features = 2
name_no_features = str(no_of_features)

max_frequency = 5; no_params = 32; bond_dim = 100; cutoff = 7; to_add = ''

name_max_freq = str(max_frequency)
name_no_params = str(no_params)
name_bond_dim = str(bond_dim)
cutoff_name = str(cutoff)

### No. basis states per parameter
dim_basis_single_param = 3  ### (1, cos(th), sin(th))
name_dim_basis_param = str(dim_basis_single_param)

Loads the pickled per-model-draw results (MSE training curves, normalized FIM spectra and effective dimensions for the full and cutoff models) saved by `train_and_FIM_biased.py` and `train_and_FIM_UNbiased.py`, for every decay_exp in decay_exp_vec, aggregating them into `dict_fullbiased_all` and `dict_unbiased_all` keyed by decay-exponent name.

In [ ]:
### ---------------------------------------------------------------------------------------- ###
## ---------------------------------------- Load data --------------------------------------- ##
### ---------------------------------------------------------------------------------------- ###

name_end = ('_BondDim' + name_bond_dim + '_Nfeatures' + name_no_features + '_MaxFreq' + name_max_freq + 
            '_Nparams' + name_no_params + '_DimBasisParam' + name_dim_basis_param + '_' + name_data_gen_fullbiased + 
            '_batch' + batch_size_name + '_lr' + learning_rate_name + '_cutoff' + cutoff_name + 
            '_decayexp')
filename0 = 'dict_results' + name_end
listallfiles = [f for f in os.listdir(results_folder) if (f.startswith(filename0))]
dict_fullbiased_all = dict()
for filename in listallfiles:
    res = re.findall((filename0 + '(\S+)_modeldraw(\S+).pkl'), filename)
    decexp_name = res[0][0]
    nm_name = res[0][1]
    nm = int(nm_name)
    dict_d = dict()
    dict_d['delta_minMSE_mean'] = []
    dict_d['delta_minMSE_std'] = []
    dict_d['delta_minMSE_min'] = []
    dict_d['delta_minMSE_max'] = []
    dict_d['MSE_full_mean'] = []
    dict_d['MSE_full_std'] = []
    dict_d['MSE_full_min'] = []
    dict_d['MSE_full_max'] = []
    dict_d['MSE_cut_mean'] = []
    dict_d['MSE_cut_std'] = []
    dict_d['MSE_cut_min'] = []
    dict_d['MSE_cut_max'] = []
    dict_d['mean_normFIM_spectra_full'] = []
    dict_d['std_normFIM_spectra_full'] = []
    dict_d['min_normFIM_spectra_full'] = []
    dict_d['max_normFIM_spectra_full'] = []
    dict_d['norm_eff_dim_full'] = []
    dict_d['mean_normFIM_spectra_cut'] = []
    dict_d['std_normFIM_spectra_cut'] = []
    dict_d['min_normFIM_spectra_cut'] = []
    dict_d['max_normFIM_spectra_cut'] = []
    dict_d['norm_eff_dim_cut'] = []
    dict_fullbiased_all[decexp_name] = dict_d
for filename in listallfiles:
    res = re.findall((filename0 + '(\S+)_modeldraw(\S+).pkl'), filename)
    decexp_name = res[0][0]
    nm_name = res[0][1]
    nm = int(nm_name)
    path_file = os.path.join(results_folder, filename)
    with open(path_file, 'rb') as f:
        dict_model = pickle.load(f)
    dict_d = dict_fullbiased_all[decexp_name]
    dict_d['delta_minMSE_mean'].append(np.mean(dict_model['delta_minMSE_full_m_cut']))
    dict_d['delta_minMSE_std'].append(np.std(dict_model['delta_minMSE_full_m_cut']))
    dict_d['delta_minMSE_min'].append(np.min(dict_model['delta_minMSE_full_m_cut']))
    dict_d['delta_minMSE_max'].append(np.max(dict_model['delta_minMSE_full_m_cut']))
    dict_d['MSE_full_mean'].append(dict_model['MSE_full_mean'])
    dict_d['MSE_full_std'].append(dict_model['MSE_full_std'])
    dict_d['MSE_full_min'].append(dict_model['MSE_full_min'])
    dict_d['MSE_full_max'].append(dict_model['MSE_full_max'])
    dict_d['MSE_cut_mean'].append(dict_model['MSE_cut_mean'])
    dict_d['MSE_cut_std'].append(dict_model['MSE_cut_std'])
    dict_d['MSE_cut_min'].append(dict_model['MSE_cut_min'])
    dict_d['MSE_cut_max'].append(dict_model['MSE_cut_max'])
    dict_d['mean_normFIM_spectra_full'].append(dict_model['mean_normFIM_spectra_full'])
    dict_d['std_normFIM_spectra_full'].append(dict_model['std_normFIM_spectra_full'])
    dict_d['min_normFIM_spectra_full'].append(dict_model['min_normFIM_spectra_full'])
    dict_d['max_normFIM_spectra_full'].append(dict_model['max_normFIM_spectra_full'])
    dict_d['norm_eff_dim_full'].append(dict_model['norm_eff_dim_full'])
    dict_d['mean_normFIM_spectra_cut'].append(dict_model['mean_normFIM_spectra_cut'])
    dict_d['std_normFIM_spectra_cut'].append(dict_model['std_normFIM_spectra_cut'])
    dict_d['min_normFIM_spectra_cut'].append(dict_model['min_normFIM_spectra_cut'])
    dict_d['max_normFIM_spectra_cut'].append(dict_model['max_normFIM_spectra_cut'])
    dict_d['norm_eff_dim_cut'].append(dict_model['norm_eff_dim_cut'])
    dict_fullbiased_all[decexp_name] = dict_d


name_end = ('_BondDim' + name_bond_dim + '_Nfeatures' + name_no_features + '_MaxFreq' + name_max_freq + 
            '_Nparams' + name_no_params + '_DimBasisParam' + name_dim_basis_param + '_' + name_data_gen_unbiased + 
            '_batch' + batch_size_name + '_lr' + learning_rate_name + '_cutoff' + cutoff_name + 
            '_decayexp')
filename0 = 'dict_results' + name_end
listallfiles = [f for f in os.listdir(results_folder) if (f.startswith(filename0))]
dict_unbiased_all = dict()
for filename in listallfiles:
    res = re.findall((filename0 + '(\S+)_modeldraw(\S+).pkl'), filename)
    decexp_name = res[0][0]
    nm_name = res[0][1]
    nm = int(nm_name)
    dict_d = dict()
    dict_d['delta_minMSE_mean'] = []
    dict_d['delta_minMSE_std'] = []
    dict_d['delta_minMSE_min'] = []
    dict_d['delta_minMSE_max'] = []
    dict_d['MSE_full_mean'] = []
    dict_d['MSE_full_std'] = []
    dict_d['MSE_full_min'] = []
    dict_d['MSE_full_max'] = []
    dict_d['MSE_cut_mean'] = []
    dict_d['MSE_cut_std'] = []
    dict_d['MSE_cut_min'] = []
    dict_d['MSE_cut_max'] = []
    dict_d['mean_normFIM_spectra_full'] = []
    dict_d['std_normFIM_spectra_full'] = []
    dict_d['min_normFIM_spectra_full'] = []
    dict_d['max_normFIM_spectra_full'] = []
    dict_d['norm_eff_dim_full'] = []
    dict_d['mean_normFIM_spectra_cut'] = []
    dict_d['std_normFIM_spectra_cut'] = []
    dict_d['min_normFIM_spectra_cut'] = []
    dict_d['max_normFIM_spectra_cut'] = []
    dict_d['norm_eff_dim_cut'] = []
    dict_unbiased_all[decexp_name] = dict_d
for filename in listallfiles:
    res = re.findall((filename0 + '(\S+)_modeldraw(\S+).pkl'), filename)
    decexp_name = res[0][0]
    nm_name = res[0][1]
    nm = int(nm_name)
    path_file = os.path.join(results_folder, filename)
    with open(path_file, 'rb') as f:
        dict_model = pickle.load(f)
    dict_d = dict_unbiased_all[decexp_name]
    dict_d['delta_minMSE_mean'].append(np.mean(dict_model['delta_minMSE_full_m_cut']))
    dict_d['delta_minMSE_std'].append(np.std(dict_model['delta_minMSE_full_m_cut']))
    dict_d['delta_minMSE_min'].append(np.min(dict_model['delta_minMSE_full_m_cut']))
    dict_d['delta_minMSE_max'].append(np.max(dict_model['delta_minMSE_full_m_cut']))
    dict_d['MSE_full_mean'].append(dict_model['MSE_full_mean'])
    dict_d['MSE_full_std'].append(dict_model['MSE_full_std'])
    dict_d['MSE_full_min'].append(dict_model['MSE_full_min'])
    dict_d['MSE_full_max'].append(dict_model['MSE_full_max'])
    dict_d['MSE_cut_mean'].append(dict_model['MSE_cut_mean'])
    dict_d['MSE_cut_std'].append(dict_model['MSE_cut_std'])
    dict_d['MSE_cut_min'].append(dict_model['MSE_cut_min'])
    dict_d['MSE_cut_max'].append(dict_model['MSE_cut_max'])
    dict_d['mean_normFIM_spectra_full'].append(dict_model['mean_normFIM_spectra_full'])
    dict_d['std_normFIM_spectra_full'].append(dict_model['std_normFIM_spectra_full'])
    dict_d['min_normFIM_spectra_full'].append(dict_model['min_normFIM_spectra_full'])
    dict_d['max_normFIM_spectra_full'].append(dict_model['max_normFIM_spectra_full'])
    dict_d['norm_eff_dim_full'].append(dict_model['norm_eff_dim_full'])
    dict_d['mean_normFIM_spectra_cut'].append(dict_model['mean_normFIM_spectra_cut'])
    dict_d['std_normFIM_spectra_cut'].append(dict_model['std_normFIM_spectra_cut'])
    dict_d['min_normFIM_spectra_cut'].append(dict_model['min_normFIM_spectra_cut'])
    dict_d['max_normFIM_spectra_cut'].append(dict_model['max_normFIM_spectra_cut'])
    dict_d['norm_eff_dim_cut'].append(dict_model['norm_eff_dim_cut'])
    dict_unbiased_all[decexp_name] = dict_d

MSE training curves for the biased scenario: for one model draw at a fixed decay_exp, plots MSE vs. epoch (mean, with min-max band over the training tests) for the full and cutoff model, annotated with each model's estimated normalized effective dimension, and saves the figure.

In [ ]:
n_draw = 0
dec_exp_name = '2p0'
dict_data = dict_fullbiased_all

fs = 28
figsize = (8,4)

plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

ned_f = dict_data[dec_exp_name]['norm_eff_dim_full'][n_draw]
y_f = dict_data[dec_exp_name]['MSE_full_mean'][n_draw]
min_y_f = dict_data[dec_exp_name]['MSE_full_min'][n_draw]
max_y_f = dict_data[dec_exp_name]['MSE_full_max'][n_draw]

ned_c = dict_data[dec_exp_name]['norm_eff_dim_cut'][n_draw]
y_c = dict_data[dec_exp_name]['MSE_cut_mean'][n_draw]
min_y_c = dict_data[dec_exp_name]['MSE_cut_min'][n_draw]
max_y_c = dict_data[dec_exp_name]['MSE_cut_max'][n_draw]

x_f = np.arange(1, len(y_f) + 1)
ax.plot(x_f, y_f, 'o-', color='tab:blue', markersize=7, linewidth=3, label=r'$\mathrm{full:\;}\hat{d}_{\mathrm{eff}}='+str(ned_f)[:5]+'$')
ax.fill_between(x_f, min_y_f, max_y_f, alpha=0.3, edgecolor='tab:blue', facecolor='tab:blue', linewidth=0.5)

x_c = np.arange(1, len(y_c) + 1)
ax.plot(x_c, y_c, 'o-', color='tab:orange', markersize=7, linewidth=3, label=r'$\mathrm{cut:\;}\hat{d}_{\mathrm{eff}}='+str(ned_c)[:5]+'$')
ax.fill_between(x_c, min_y_c, max_y_c, alpha=0.3, edgecolor='tab:orange', facecolor='tab:orange', linewidth=0.5)

#ax.legend(fontsize=22)
ax.legend(loc='lower left', fontsize=26)
ax.set_xlabel(r'$\mathrm{epoch}$', fontsize=fs)
ax.set_ylabel(r'$\mathrm{MSE}$', fontsize=fs)
ax.set_xscale('log')
ax.set_yscale('log')

In [ ]:
fig.savefig('MSE_training_fullbiased_full_and_cut' + to_add + '.png', bbox_inches='tight')
fig.savefig('MSE_training_fullbiased_full_and_cut' + to_add + '.pdf', bbox_inches='tight')

MSE training curves for the unbiased scenario: same plot as above (MSE vs. epoch, mean and min-max band, with effective dimensions annotated), but for one model draw of the unbiased data-generating experiment, saved to disk.

In [ ]:
n_draw = 5
dec_exp_name = '2p0'
dict_data = dict_unbiased_all

fs = 28
figsize = (8,4)

plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

ned_f = dict_data[dec_exp_name]['norm_eff_dim_full'][n_draw]
y_f = dict_data[dec_exp_name]['MSE_full_mean'][n_draw]
min_y_f = dict_data[dec_exp_name]['MSE_full_min'][n_draw]
max_y_f = dict_data[dec_exp_name]['MSE_full_max'][n_draw]

ned_c = dict_data[dec_exp_name]['norm_eff_dim_cut'][n_draw]
y_c = dict_data[dec_exp_name]['MSE_cut_mean'][n_draw]
min_y_c = dict_data[dec_exp_name]['MSE_cut_min'][n_draw]
max_y_c = dict_data[dec_exp_name]['MSE_cut_max'][n_draw]

x_f = np.arange(1, len(y_f) + 1)
ax.plot(x_f, y_f, 'o-', color='tab:blue', markersize=7, linewidth=3, label=r'$\mathrm{full:\;}\hat{d}_{\mathrm{eff}}='+str(ned_f)[:5]+'$')
ax.fill_between(x_f, min_y_f, max_y_f, alpha=0.3, edgecolor='tab:blue', facecolor='tab:blue', linewidth=0.5)

x_c = np.arange(1, len(y_c) + 1)
ax.plot(x_c, y_c, 'o-', color='tab:orange', markersize=7, linewidth=3, label=r'$\mathrm{cut:\;}\hat{d}_{\mathrm{eff}}='+str(ned_c)[:5]+'$')
ax.fill_between(x_c, min_y_c, max_y_c, alpha=0.3, edgecolor='tab:orange', facecolor='tab:orange', linewidth=0.5)

#ax.legend(fontsize=22)
ax.legend(loc='lower left', fontsize=26)
ax.set_xlabel(r'$\mathrm{epoch}$', fontsize=fs)
ax.set_ylabel(r'$\mathrm{MSE}$', fontsize=fs)
ax.set_xscale('log')
ax.set_yscale('log')

In [ ]:
fig.savefig('MSE_training_unbiased_full_and_cut' + to_add + '.png', bbox_inches='tight')
fig.savefig('MSE_training_unbiased_full_and_cut' + to_add + '.pdf', bbox_inches='tight')

Delta_{f-c}MSE_min vs. effective-dimension gap, biased and unbiased models overlaid: for every decay_exp and model draw, plots Delta_{f-c}MSE_min = MSE_full_min - MSE_cut_min against d_eff^full - d_eff^cut, for the biased (one color) and unbiased (other color) scenarios, with a zero reference line, then saves the figure. This reproduces the paper's key finding that Delta is positive for biased models (low ED trains better) and negative for unbiased models (high ED trains better).

In [ ]:
fs = 28
figsize = (9,6)

cmap = matplotlib.colormaps['cividis']

plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

deltaMSE_all_fullbiased = []
nED_full_all_fullbiased = []
nED_cut_all_fullbiased = []
for decay_exp_name in dict_fullbiased_all:
    dict_dec = dict_fullbiased_all[decay_exp_name]
    nED_full = np.asarray(dict_dec['norm_eff_dim_full'])
    nED_cut = np.asarray(dict_dec['norm_eff_dim_cut'])
    deltaMSE_mean = np.asarray(dict_dec['delta_minMSE_mean'])
    deltaMSE_all_fullbiased.append(deltaMSE_mean)
    nED_full_all_fullbiased.append(nED_full)
    nED_cut_all_fullbiased.append(nED_cut)
nED_full_fullbiased = np.concatenate(nED_full_all_fullbiased)
nED_cut_fullbiased = np.concatenate(nED_cut_all_fullbiased)
deltaMSE_all_fullbiased = np.concatenate(deltaMSE_all_fullbiased)
y = deltaMSE_all_fullbiased
x = nED_full_fullbiased - nED_cut_fullbiased
ax.plot(x, y, 'D', color=cmap(0.0), markersize=7, markeredgewidth=1.0, markeredgecolor='k')
yy1 = copy.deepcopy(y)

deltaMSE_all_unbiased = []
finalMSE_cut_all_unbiased = []
nED_full_all_unbiased = []
nED_cut_all_unbiased = []
for decay_exp_name in dict_unbiased_all:
    dict_dec = dict_unbiased_all[decay_exp_name]
    nED_full = np.asarray(dict_dec['norm_eff_dim_full'])
    nED_cut = np.asarray(dict_dec['norm_eff_dim_cut'])
    deltaMSE_mean = np.asarray(dict_dec['delta_minMSE_mean'])
    deltaMSE_all_unbiased.append(deltaMSE_mean)
    nED_full_all_unbiased.append(nED_full)
    nED_cut_all_unbiased.append(nED_cut)
nED_full_unbiased = np.concatenate(nED_full_all_unbiased)
nED_cut_unbiased = np.concatenate(nED_cut_all_unbiased)
deltaMSE_all_unbiased = np.concatenate(deltaMSE_all_unbiased)
y = deltaMSE_all_unbiased
x = nED_full_unbiased - nED_cut_unbiased
ax.plot(x, y, 'D', color=cmap(1.0), markersize=7, markeredgewidth=1.0, markeredgecolor='k')
yy2 = copy.deepcopy(y)

xx = np.arange(0.0, 0.41, 0.001)
ax.plot(xx, np.zeros(len(xx)), '--', color='red', linewidth=3)

#ax.legend(fontsize=22)
ax.set_xlabel(r'$\hat{d}^{\mathrm{full}}_{\mathrm{eff}}-\hat{d}^{\mathrm{cut}}_{\mathrm{eff}}$', fontsize=fs)
ax.set_ylabel(r'$\Delta_{\mathrm{f-c}}\,\mathrm{MSE}_{\mathrm{min}}$', fontsize=fs)
#ax.set_xscale('log')
#ax.set_yscale('log')
ax.set_yscale('symlog', linthresh=1.0)
#ax.set_ylim([1.0e-12, 1.0e03])

In [ ]:
fig.savefig('DeltaMSEmin_vs_DeltaEffDim_biased_and_unbiased' + to_add + '.png', bbox_inches='tight')
fig.savefig('DeltaMSEmin_vs_DeltaEffDim_biased_and_unbiased' + to_add + '.pdf', bbox_inches='tight')